# ClaimCheck V3: training the fact-checking model

This notebook fine-tunes **DeBERTa-v3-base** to read a *claim* and some *evidence* and decide:
**SUPPORTS**, **REFUTES** or **NOT ENOUGH INFO**.

Run the cells **one at a time, from top to bottom** (click a cell, then Shift + Enter).

Before you start: **Runtime → Change runtime type → T4 GPU → Save**.

## 1. Check the GPU and install two extra libraries
`nvidia-smi` should show a **Tesla T4**. If it says "command not found", the GPU is not switched on (see above).
`peft` gives us LoRA; `sentencepiece` is needed by DeBERTa's tokenizer. `torchao` is removed because Colab's version clashes with peft.

In [ ]:
!nvidia-smi
!pip uninstall -y -q torchao   # Colab's old torchao clashes with peft; we don't need it
!pip install -q peft sentencepiece

## 2. Log in to Hugging Face
Paste your Hugging Face token (a **Write** token) when asked. This lets the notebook upload the trained model to your account at the end.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Settings
Change `HF_USERNAME` to **your** Hugging Face username.

In [ ]:
HF_USERNAME = "hiir21"          
HF_REPO = f"{HF_USERNAME}/claimcheck-deberta-v3-fever"

BASE_MODEL = "microsoft/deberta-v3-base"  # the pre-trained model we start from
PER_LABEL = 10_000                        # training examples per label (30,000 in total)
MAX_LEN = 256                             # longest input in tokens; longer evidence is cut
SEED = 42

## 4. Upload your exam file, so we can make sure none of it is used for training
Click **Choose Files** and select `data/fever_test.jsonl` from your claimcheck folder.

Training on exam questions would be cheating ("data leakage"): the model would just remember the answers.

In [ ]:
import json
from google.colab import files

uploaded = files.upload()
content = next(iter(uploaded.values()))  # works even if Colab renames it "fever_test (2).jsonl"
test = [json.loads(line) for line in content.decode("utf-8").splitlines() if line.strip()]
test_ids = {c["id"] for c in test}
test_claims = {c["claim"].strip().lower() for c in test}
print(f"Exam: {len(test)} claims will be kept out of training.")

## 5. Load the training data (FEVER in NLI format)
Each row has a **premise** (the claim), a **hypothesis** (Wikipedia evidence text) and a **label**:
0 = entailment (SUPPORTS), 1 = neutral (NOT ENOUGH INFO), 2 = contradiction (REFUTES).

We use only the **train** split: the other splits overlap with FEVER's dev set, which our exam comes from.

In [ ]:
from datasets import load_dataset, concatenate_datasets

raw = load_dataset("pietrolesci/nli_fever", split="train")
print(raw)
print(raw[0])

# Remove empty evidence and anything that overlaps with the exam
clean = raw.filter(lambda x: len(x["hypothesis"].strip()) > 0
                   and x["cid"] not in test_ids
                   and x["premise"].strip().lower() not in test_claims)
print(f"After cleaning: {len(clean)} rows (removed {len(raw) - len(clean)})")

# Take the same number of examples from each label, so the model doesn't favour one answer
parts = [clean.filter(lambda x, l=l: x["label"] == l).shuffle(seed=SEED).select(range(PER_LABEL))
         for l in range(3)]
data = concatenate_datasets(parts).shuffle(seed=SEED)

# Keep 2,000 aside to check the model during training (validation set)
split = data.train_test_split(test_size=2000, seed=SEED)
train_ds, val_ds = split["train"], split["test"]
print(f"Training: {len(train_ds)}, validation: {len(val_ds)}")

## 6. Tokenise: turn text into numbers the model understands
The claim and the evidence go in **together** as one pair. If it's too long, only the evidence is cut.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(batch["premise"], batch["hypothesis"], truncation="only_second", max_length=MAX_LEN)

keep = ["input_ids", "attention_mask", "label"]
train_tok = train_ds.map(tokenize, batched=True)
val_tok = val_ds.map(tokenize, batched=True)
train_tok = train_tok.remove_columns([c for c in train_tok.column_names if c not in keep])
val_tok = val_tok.remove_columns([c for c in val_tok.column_names if c not in keep])
print(train_tok)

## 7. Load DeBERTa and add LoRA
The pre-trained model already understands English. We add a small 3-way classification layer on top,
and LoRA: small trainable "adapter" matrices inside the attention layers. Only these and the classifier
are trained; the original weights stay frozen. Watch the line that says how many parameters are trainable.

In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, TaskType, get_peft_model

id2label = {0: "SUPPORTS", 1: "NOT ENOUGH INFO", 2: "REFUTES"}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=3, id2label=id2label, label2id=label2id)
model = model.float()  # keep weights in 32-bit: newer versions may load them in 16-bit, which breaks fp16 training

lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.1,
                  target_modules=["query_proj", "key_proj", "value_proj"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 8. Train
About **30 to 45 minutes** on a free T4. Every 300 steps it checks itself on the validation set:
watch **eval_macro_f1** go up. Keep this browser tab open (Colab may disconnect an idle tab).

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro")}

args = TrainingArguments(
    output_dir="checker",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-4,          # LoRA usually needs a higher learning rate than full fine-tuning
    warmup_steps=100,            # start with a small learning rate for the first 100 steps
    weight_decay=0.01,
    fp16=True,                   # half-precision: faster on the T4 GPU
    eval_strategy="steps",
    eval_steps=300,
    logging_steps=100,
    save_strategy="no",
    report_to="none",
    seed=SEED,
)

trainer = Trainer(model=model, args=args, train_dataset=train_tok, eval_dataset=val_tok,
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
trainer.train()
print(trainer.evaluate())

## 9. Quick sanity check on three made-up examples

In [ ]:
import torch

def check(claim, evidence):
    enc = tokenizer(claim, evidence, truncation="only_second", max_length=MAX_LEN, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits, dim=-1)[0].tolist()
    best = int(np.argmax(probs))
    return id2label[best], {id2label[i]: round(p, 3) for i, p in enumerate(probs)}

ev = "Eiffel Tower . The Eiffel Tower is a lattice tower on the Champ de Mars in Paris, France."
print(check("The Eiffel Tower is in Paris.", ev))     # expect SUPPORTS
print(check("The Eiffel Tower is in Berlin.", ev))    # expect REFUTES
print(check("The Eiffel Tower is painted red.", ev))  # expect NOT ENOUGH INFO

## 10. Save the model to your Hugging Face account
LoRA's small adapters are merged back into the model, so it can be loaded like any normal model.
When this finishes, open the link it prints: that page goes on your CV.

In [ ]:
merged = model.merge_and_unload()
merged.push_to_hub(HF_REPO, commit_message="ClaimCheck V3 checker: DeBERTa-v3-base + LoRA on FEVER-NLI")
tokenizer.push_to_hub(HF_REPO)
print(f"Done: https://huggingface.co/{HF_REPO}")